# Carnatify — raga feature extraction on Colab Pro (GPU) — **v3**

Runs the production-matched Demucs+pyin pipeline (`raga_v2_pipeline.py`) over the
shankarkrish/archive.org concert downloads, so the raga classifier can be retrained
on features that match live inference exactly.

**Why v3:** the v1 features were normalized by a median-F0 "tonic" that matched the
true Sa only 10% of the time (measured against Saraga annotations) — that rotated
every feature vector randomly and capped CV accuracy at 9%. v3 estimates the tonic
from the tambura drone in the *mixed* audio (essentia `TonicIndianArtMusic`, 87%
within ±50 cents) before Demucs strips it. Output goes to a fresh
`carnatify_raga_cache_v3/` folder; the old cache is obsolete.

**Before running:**
1. Runtime → Change runtime type → **GPU**.
2. The same `carnatify_concert_audio.zip` you already uploaded to Drive is reused —
   nothing to re-upload.
3. Run all cells. Features accumulate in `MyDrive/carnatify_raga_cache_v3/` — one
   `.npz` per track — so the run is **resumable across Colab sessions**.

**After it finishes:** download the `carnatify_raga_cache_v3` folder from Drive
(right-click → Download) and tell Claude — it handles extraction, retraining, and
evaluation locally.

Time estimate: similar to v1 (~45-90 s/track with 4 workers; tonic estimation adds
a few seconds per track). Safe to stop and re-run anytime.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip -q install demucs soundfile librosa essentia
import essentia.standard as es
assert hasattr(es, 'TonicIndianArtMusic'), 'essentia broken'
print('essentia OK — tonic estimation will work')
!git clone -q https://github.com/ShyamRavidath/carnatify.git /content/carnatify
!unzip -q -n /content/drive/MyDrive/carnatify_concert_audio.zip -d /content/
!ls /content/concert_audio | head -3

In [ ]:
import multiprocessing as mp
import sys
import numpy as np
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from concurrent.futures.process import BrokenProcessPool

AUDIO_ROOT = Path('/content/concert_audio')
CACHE_DIR = Path('/content/drive/MyDrive/carnatify_raga_cache_v3')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
EXCLUDED_LABELS = {'Rāgamālika'}  # a form (garland of ragas), not a raga label

# spawn workers must IMPORT the worker fn from a real module — a function
# defined in a notebook cell is unimportable in spawned children (they die
# instantly with BrokenProcessPool). So write it to a file.
Path('/content/extract_worker.py').write_text('''
import sys
sys.path.insert(0, "/content/carnatify")
from pathlib import Path
from raga_v2_pipeline import process_track
CACHE_DIR = Path("/content/drive/MyDrive/carnatify_raga_cache_v3")
def run(job):
    track_id, raga, path = job
    try:
        return track_id, process_track(track_id, raga, path, CACHE_DIR)
    except Exception as exc:
        return track_id, {"status": f"error: {exc}"}
''')
sys.path.insert(0, '/content')
from extract_worker import run

# Prune stale entries: v1 npz (no tonic_method) and median_f0 fallbacks carry
# random tonic rotations — remove them.
pruned = 0
for p in list(CACHE_DIR.glob('*.npz')):
    try:
        d = np.load(p, allow_pickle=True)
        ok = 'tonic_method' in d.files and str(d['tonic_method']) != 'median_f0'
    except Exception:
        ok = False
    if not ok:
        p.unlink()
        pruned += 1
print(f'pruned {pruned} stale/median_f0 npz from cache')

jobs = []
for raga_dir in sorted(AUDIO_ROOT.iterdir()):
    if not raga_dir.is_dir() or raga_dir.name in EXCLUDED_LABELS:
        continue
    for mp3 in sorted(raga_dir.glob('*.mp3')):
        # must match train_raga_v2_archive.py's naming so local + Colab caches merge
        jobs.append((f'archive__{raga_dir.name}__{mp3.stem}', raga_dir.name, str(mp3)))
print(f'{len(jobs)} tracks total')

tonic_methods = {}
round_num = 0
while True:
    already = {p.stem for p in CACHE_DIR.glob('*.npz')}
    todo = [j for j in jobs if j[0] not in already]
    if not todo:
        break
    round_num += 1
    print(f'round {round_num}: {len(already)} cached, {len(todo)} to process', flush=True)
    done = 0
    try:
        with ProcessPoolExecutor(max_workers=2, mp_context=mp.get_context('spawn')) as pool:
            for fut in as_completed([pool.submit(run, j) for j in todo]):
                track_id, result = fut.result()
                done += 1
                status = (result or {}).get('status', 'skipped (too short / no pitch)')
                tm = (result or {}).get('tonic_method')
                if tm:
                    tonic_methods[tm] = tonic_methods.get(tm, 0) + 1
                if done == 1 and tm == 'median_f0':
                    raise RuntimeError('First track fell back to median_f0 — essentia broken in workers. STOP.')
                if status.startswith('error') or done % 10 == 0:
                    print(f'[{done}/{len(todo)}] {track_id}: {status}', flush=True)
        break
    except BrokenProcessPool:
        made_progress = len({p.stem for p in CACHE_DIR.glob('*.npz')}) > len(already)
        if not made_progress:
            raise RuntimeError('Pool died with ZERO progress — restarting will not help. '
                               'Check RAM (Runtime > View resources) and worker importability.')
        print('pool died after partial progress — restarting from cache', flush=True)

print(f'\\nfinished; cache has {len(list(CACHE_DIR.glob("*.npz")))} tracks')
print('tonic methods:', tonic_methods, '— must be essentia; median_f0 means broken run')